In [ ]:
# @title Lib Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# @title Data Fetch
import kagglehub
path = kagglehub.dataset_download("stackoverflow/stack-overflow-2023-developers-survey")
print("Path to dataset files:", path)
df = pd.read_csv('/root/.cache/kagglehub/datasets/stackoverflow/stack-overflow-2023-developers-survey/versions/1/survey_results_public.csv')

In [ ]:
# @title Basic Data Clean
def load_and_clean_data(df):
    # Select essential features only
    features = ['YearsCodePro', 'EdLevel', 'DevType', 'OrgSize',
               'Country', 'Employment', 'Age', 'ConvertedCompYearly']

    df = df[features].copy()

    # Remove extreme outliers
    df = df[(df['ConvertedCompYearly'] >= 30000) &
            (df['ConvertedCompYearly'] <= 500000)]

    return df

In [ ]:
# @title Feature Preprocessing | Data Augmentation
def preprocess_features(df):

    # Convert experience to numeric
    def convert_experience(value):
        if pd.isna(value):
            return np.nan
        value_str = str(value).lower()
        if 'less than' in value_str or '<' in value_str:
            return 0
        return pd.to_numeric(value, errors='coerce')

    df['YearsCodePro'] = df['YearsCodePro'].apply(convert_experience)

    # Simplify education levels
    def simplify_education(ed):
        if pd.isna(ed):
            return 'No Degree'
        ed = str(ed).lower()
        if 'master' in ed or 'graduate' in ed:
            return 'Masters'
        elif 'bachelor' in ed:
            return 'Bachelors'
        elif 'doctoral' in ed or 'phd' in ed:
            return 'PhD'
        else:
            return 'Other'

    df['EdLevel'] = df['EdLevel'].apply(simplify_education)

    # Simplify developer types
    def simplify_devtype(devtype):
        if pd.isna(devtype):
            return 'Other'
        devtype = str(devtype).lower()
        if 'data scientist' in devtype or 'machine learning' in devtype:
            return 'Data Science'
        elif 'manager' in devtype or 'lead' in devtype:
            return 'Management'
        elif 'backend' in devtype or 'full-stack' in devtype:
            return 'Backend'
        elif 'frontend' in devtype or 'mobile' in devtype:
            return 'Frontend'
        else:
            return 'Other'

    df['DevType'] = df['DevType'].apply(simplify_devtype)

    # Keeping only top 10 countries
    top_countries = df['Country'].value_counts().head(10).index
    df['Country'] = df['Country'].where(df['Country'].isin(top_countries), 'Other')

    # Simplify employment
    def simplify_employment(emp):
        if pd.isna(emp):
            return 'Other'
        emp = str(emp).lower()
        if 'full-time' in emp:
            return 'Full-time'
        elif 'part-time' in emp:
            return 'Part-time'
        elif 'student' in emp:
            return 'Student'
        else:
            return 'Other'

    df['Employment'] = df['Employment'].apply(simplify_employment)

    return df

In [ ]:
# @title Pipe Line
def create_model_pipeline():
    """Create preprocessing and model pipeline"""

    # Defineing column types
    numeric_features = ['YearsCodePro']
    categorical_features = ['EdLevel', 'DevType', 'OrgSize', 'Country', 'Employment', 'Age']

    # preprocessing pipeline
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())
            ]), numeric_features),
            ('cat', Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
            ]), categorical_features)
        ]
    )

    # full pipeline with model
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', GradientBoostingRegressor(
            n_estimators=100,
            max_depth=6,
            learning_rate=0.1,
            random_state=42
        ))
    ])

    return pipeline

def evaluate_model(y_true, y_pred):
    """Calculate and display evaluation metrics"""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = np.mean(np.abs(y_true - y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    print(f"RMSE: ${rmse:,.0f}")
    print(f"MAE: ${mae:,.0f}")
    print(f"MAPE: {mape:.1f}%")

    return rmse

def plot_results(y_true, y_pred):
    """Plot actual vs predicted values"""
    plt.figure(figsize=(10, 6))
    plt.scatter(y_true, y_pred, alpha=0.5)
    plt.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', lw=2)
    plt.xlabel('Actual Salary')
    plt.ylabel('Predicted Salary')
    plt.title('Actual vs Predicted Salaries')
    plt.grid(True, alpha=0.3)
    plt.show()

def main(df):
    """Main execution function"""
    print("Loading and cleaning data...")
    df_clean = load_and_clean_data(df)

    print("Preprocessing features...")
    df_processed = preprocess_features(df_clean)

    # Prepare features and target
    X = df_processed.drop('ConvertedCompYearly', axis=1)
    y = df_processed['ConvertedCompYearly']

    print(f"Dataset shape: {X.shape}")
    print(f"Target range: ${y.min():,.0f} - ${y.max():,.0f}")

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Create and train model
    print("Training model...")
    pipeline = create_model_pipeline()
    pipeline.fit(X_train, y_train)

    # Make predictions
    y_pred = pipeline.predict(X_test)

    # Evaluate
    print("\nModel Performance:")
    rmse = evaluate_model(y_test, y_pred)

    # Plot results
    plot_results(y_test, y_pred)

    # Feature importance (approximate)
    try:
        feature_names = (pipeline.named_steps['preprocessor']
                        .named_transformers_['cat']
                        .named_steps['encoder']
                        .get_feature_names_out())
        importance = pipeline.named_steps['regressor'].feature_importances_

        # Show top 10 features
        importance_df = pd.DataFrame({
            'feature': list(feature_names) + ['YearsCodePro'],
            'importance': importance
        }).sort_values('importance', ascending=False)

        print("\nTop 10 Most Important Features:")
        print(importance_df.head(10))

    except Exception as e:
        print("Could not extract feature importance:", e)

    return pipeline, rmse

In [ ]:
# @title Model Run
model, rmse = main(df)